In [ ]:
# | default_exp schema_validator

In [ ]:
# | export
from __future__ import annotations
import json
import re
from typing import Any
import requests
from bs4 import BeautifulSoup

In [ ]:
# | export

# Google Rich Results spec: required and recommended fields per type
# Source: http://developers.google.com/search/docs/appearance/structured-data/search-gallery
GOOGLE_SPEC: dict[str, dict[str, list[str]]] = {
    "Article": {
        "required": ["@type", "headline", "image", "datePublished", "author"],
        "recommended": ["dateModified", "author.@type", "author.name", "publisher", "publisher.@type",
                        "publisher.name"],
    },
    "NewsArticle": {
        "required": ["@type", "headline", "image", "datePublished", "author"],
        "recommended": ["dateModified", "author.name", "publisher", "publisher.name"],
    },
    "BlogPosting": {
        "required": ["@type", "headline", "image", "datePublished", "author"],
        "recommended": ["dateModified", "author.name", "publisher", "publisher.name"],
    },
    "BreadcrumbList": {
        "required": ["@type", "itemListElement"],
        "recommended": ["itemListElement.@type", "itemListElement.position", "itemListElement.name",
                        "itemListElement.item"],
    },
    "Course": {
        "required": ["@type", "name", "description", "provider"],
        "recommended": ["provider.@type", "provider.name", "url", "hasCourseInstance"],
    },
    "Dataset": {
        "required": ["@type", "name", "description"],
        "recommended": ["url", "sameAs", "keywords", "creator", "license", "temporalCoverage", "spatialCoverage"],
    },
    "DiscussionForumPosting": {
        "required": ["@type", "headline", "url", "author"],
        "recommended": ["datePublished", "text", "author.name"],
    },
    "EmployerAggregateRating": {
        "required": ["@type", "itemReviewed", "ratingValue", "bestRating", "worstRating", "ratingCount"],
        "recommended": ["itemReviewed.name", "itemReviewed.sameAs"],
    },
    "Event": {
        "required": ["@type", "name", "startDate", "location"],
        "recommended": ["endDate", "description", "image", "organizer", "performer", "eventStatus",
                        "eventAttendanceMode", "offers"],
    },
    "FAQPage": {
        "required": ["@type", "mainEntity"],
        "recommended": ["mainEntity.@type", "mainEntity.name", "mainEntity.acceptedAnswer",
                        "mainEntity.acceptedAnswer.text"],
    },
    "ImageObject": {
        "required": ["@type", "contentUrl"],
        "recommended": ["license", "acquireLicensePage", "creditText", "creator", "copyrightNotice"],
    },
    "JobPosting": {
        "required": ["@type", "title", "description", "hiringOrganization", "jobLocation", "datePosted"],
        "recommended": ["validThrough", "employmentType", "baseSalary", "hiringOrganization.name",
                        "hiringOrganization.sameAs"],
    },
    "LocalBusiness": {
        "required": ["@type", "name", "address"],
        "recommended": ["address.@type", "address.streetAddress", "address.addressLocality", "address.addressCountry",
                        "telephone", "url", "openingHours", "geo", "priceRange", "image"],
    },
    "MathSolver": {
        "required": ["@type", "name", "potentialAction"],
        "recommended": ["potentialAction.@type", "potentialAction.mathExpression"],
    },
    "Movie": {
        "required": ["@type", "name"],
        "recommended": ["url", "image", "director", "dateCreated", "genre", "description", "aggregateRating"],
    },
    "Organization": {
        "required": ["@type", "name", "url", "logo"],
        "recommended": ["contactPoint", "sameAs", "address", "telephone", "foundingDate", "numberOfEmployees"],
    },
    "Product": {
        "required": ["@type", "name"],
        "recommended": ["image", "description", "sku", "brand", "offers", "aggregateRating", "review"],
    },
    "ProfilePage": {
        "required": ["@type", "mainEntity"],
        "recommended": ["mainEntity.@type", "mainEntity.name", "mainEntity.description", "mainEntity.image"],
    },
    "QAPage": {
        "required": ["@type", "mainEntity"],
        "recommended": ["mainEntity.@type", "mainEntity.name", "mainEntity.acceptedAnswer",
                        "mainEntity.suggestedAnswer"],
    },
    "Recipe": {
        "required": ["@type", "name", "image", "author", "datePublished", "description", "recipeIngredient",
                     "recipeInstructions"],
        "recommended": ["prepTime", "cookTime", "totalTime", "recipeYield", "nutrition", "aggregateRating", "video"],
    },
    "Review": {
        "required": ["@type", "itemReviewed", "reviewRating", "author"],
        "recommended": ["reviewRating.ratingValue", "reviewRating.bestRating", "author.name", "reviewBody",
                        "publisher"],
    },
    "AggregateRating": {
        "required": ["@type", "ratingValue", "reviewCount"],
        "recommended": ["bestRating", "worstRating", "itemReviewed"],
    },
    "SoftwareApplication": {
        "required": ["@type", "name", "operatingSystem", "applicationCategory"],
        "recommended": ["offers", "aggregateRating", "offers.price", "offers.priceCurrency"],
    },
    "SpeakableSpecification": {
        "required": ["@type", "cssSelector"],
        "recommended": ["xpath"],
    },
    "VacationRental": {
        "required": ["@type", "name", "description", "url"],
        "recommended": ["image", "address", "geo", "telephone", "aggregateRating", "numberOfRooms"],
    },
    "VideoObject": {
        "required": ["@type", "name", "description", "thumbnailUrl", "uploadDate"],
        "recommended": ["duration", "contentUrl", "embedUrl", "expires", "hasPart", "watchAction", "publication"],
    },
    "WebPage": {
        "required": ["@type", "name"],
        "recommended": ["url", "description", "breadcrumb", "lastReviewed"],
    },
    "WebSite": {
        "required": ["@type", "url"],
        "recommended": ["name", "potentialAction", "potentialAction.@type", "potentialAction.target",
                        "potentialAction.query-input"],
    },
    "Person": {
        "required": ["@type", "name"],
        "recommended": ["url", "image", "jobTitle", "worksFor", "sameAs"],
    },
    "ItemList": {
        "required": ["@type", "itemListElement"],
        "recommended": ["itemListElement.@type", "itemListElement.position", "itemListElement.url"],
    },
}

GOOGLE_SUPPORTED_TYPES: set[str] = set(GOOGLE_SPEC.keys())

In [ ]:
# | export
import httpx


def fetch_html(url: str,  # URL to fetch
               timeout: int = 10  # Request timeout in seconds
               ) -> tuple[str, int]:
    "Fetch raw HTML from a live URL. Returns (html, status_code)."
    try:
        resp = httpx.get(url, headers={"User-Agent": "Mozilla/5.0 (compatible; seo-rat-schema-validator/1.0)"},
                         timeout=timeout)
        return resp.text, resp.status_code
    except httpx.RequestError:
        return "", 0


In [ ]:
# | export

def _get_nested(obj: dict,  # Schema dict to traverse
                dotpath: str  # Dot-notated field path e.g. 'author.name'
                ) -> Any:
    "Get a value from a nested dict using dot notation, handling lists by checking first element."
    current = obj
    for part in dotpath.split("."):
        if isinstance(current, list): current = current[0] if current else None
        if not isinstance(current, dict): return None
        current = current.get(part)
    return current


def _field_present(schema: dict,  # Schema dict
                   field: str  # Field name or dot-notated path
                   ) -> bool:
    "Check if a field is present and non-empty in a schema dict."
    val = _get_nested(schema, field)
    return bool(val) if val is not None else False


def extract_jsonld_schemas(html: str) -> list[dict]:
    """Extract all JSON-LD schema blocks from HTML. Returns list of parsed dicts."""
    soup = BeautifulSoup(html, "html.parser")
    schemas = []
    for tag in soup.find_all("script", type="application/ld+json"):
        try:
            data = json.loads(tag.string or "")
            # Handle @graph arrays
            if isinstance(data, dict) and "@graph" in data:
                for item in data["@graph"]:
                    schemas.append({"format": "json-ld", "raw": item})
            elif isinstance(data, list):
                for item in data:
                    schemas.append({"format": "json-ld", "raw": item})
            elif isinstance(data, dict):
                schemas.append({"format": "json-ld", "raw": data})
        except (json.JSONDecodeError, TypeError):
            continue
    return schemas


def extract_microdata_schemas(html: str  # Raw HTML content
                              ) -> list[dict]:
    "Extract Microdata itemscope blocks from HTML as flat dicts."
    soup = BeautifulSoup(html, "html.parser")
    schemas = []
    for item in soup.find_all(attrs={"itemscope": True, "itemtype": True}):
        itemtype = item.get("itemtype", "")
        raw = {"@type": itemtype.split("/")[-1] if itemtype else ""}
        for prop in item.find_all(attrs={"itemprop": True}):
            if name := prop.get("itemprop"):
                raw[name] = (prop.get("content") or
                             prop.get("href") or
                             prop.get_text(strip=True))
        schemas.append({"format": "microdata", "raw": raw})
    return schemas


def extract_schemas(html: str) -> list[dict]:
    """Extract all schema blocks (JSON-LD + Microdata) from HTML."""
    return extract_jsonld_schemas(html) + extract_microdata_schemas(html)

In [ ]:
# | export

def validate_schema(schema_block: dict  # Dict with 'format' and 'raw' keys
                    ) -> dict:
    "Validate a single schema block against Google's rich result spec."
    raw = schema_block.get("raw", {})
    fmt = schema_block.get("format", "unknown")

    raw_type = raw.get("@type", "")
    if isinstance(raw_type, list): raw_type = next(iter(raw_type), "")
    schema_type = raw_type.replace("https://schema.org/", "").replace("http://schema.org/", "")

    google_supported = schema_type in GOOGLE_SUPPORTED_TYPES
    spec = GOOGLE_SPEC.get(schema_type, {"required": [], "recommended": []})

    missing_required = [f for f in spec["required"] if not _field_present(raw, f)]
    missing_recommended = [f for f in spec["recommended"] if not _field_present(raw, f)]
    fields_present = [f for f in spec["required"] + spec["recommended"] if _field_present(raw, f)]

    warning_checks = [
        (not schema_type,
         "@type is missing — Google cannot identify this schema"),
        (not google_supported and schema_type,
         f"'{schema_type}' is not a Google rich result type — no rich snippet will be generated"),
        (isinstance(raw.get("image"), str) and schema_type in ("Article", "Recipe", "VideoObject"),
         "'image' should be an ImageObject or array, not a bare string"),
        (schema_type == "Article" and isinstance(raw.get("author"), str),
         "'author' should be a Person or Organization object, not a bare string"),
        (schema_type == "LocalBusiness" and not raw.get("@id"),
         "'@id' is missing — recommended for LocalBusiness to enable knowledge panel"),
        (schema_type == "FAQPage" and isinstance(raw.get("mainEntity"), list) and not raw.get("mainEntity"),
         "'mainEntity' is empty — FAQPage needs at least one Question"),
    ]
    warnings = [msg for condition, msg in warning_checks if condition]

    return {"format": fmt, "type": schema_type or "(unknown)",
            "google_supported": google_supported,
            "is_valid": not missing_required and bool(schema_type),
            "all_fields_in_raw": list(raw.keys()),
            "fields_present_from_spec": fields_present,
            "fields_missing_required": missing_required,
            "fields_missing_recommended": missing_recommended,
            "warnings": warnings, "raw": raw}


In [ ]:
# | export

def validate_page(url: str,  # Live page URL to validate
                  timeout: int = 10  # HTTP request timeout in seconds
                  ) -> dict:
    "Fetch a live URL and validate all schema.org markup found on the page."
    html, status = fetch_html(url, timeout=timeout)
    if not html:
        return {"url": url, "fetch_status": status, "error": "Failed to fetch page",
                "schemas_found": [],
                "summary": {"total_schemas": 0, "valid_count": 0, "types_found": [], "has_google_supported": False}}
    validated = [validate_schema(s) for s in extract_schemas(html)]
    return {"url": url, "fetch_status": status, "schemas_found": validated,
            "summary": {"total_schemas": len(validated),
                        "valid_count": sum(1 for v in validated if v["is_valid"]),
                        "types_found": [v["type"] for v in validated],
                        "has_google_supported": any(v["google_supported"] for v in validated)}}

In [ ]:
# | export

def validate_pages(urls: list[str],  # List of page URLs to validate
                   timeout: int = 10  # HTTP request timeout in seconds
                   ) -> list[dict]:
    "Validate schema markup for a list of URLs."
    return [validate_page(url, timeout=timeout) for url in urls]

In [ ]:
# | hide
#| eval: false
result = validate_page("http://localhost:4321/")
print("Status:", result["fetch_status"])
print("Summary:", result["summary"])

Status: 200
Summary: {'total_schemas': 3, 'valid_count': 3, 'types_found': ['WebSite', 'LocalBusiness', 'Organization'], 'has_google_supported': True}


In [ ]:
# | hide
#| eval: false
for schema in result["schemas_found"]:
    print(f"\n--- Type: {schema['type']} ({schema['format']}) ---")
    print(f"  Google supported : {schema['google_supported']}")
    print(f"  Valid            : {schema['is_valid']}")
    print(f"  Fields in raw    : {schema['all_fields_in_raw']}")
    print(f"  Present (spec)   : {schema['fields_present_from_spec']}")
    print(f"  Missing required : {schema['fields_missing_required']}")
    print(f"  Missing recmd    : {schema['fields_missing_recommended']}")
    if schema['warnings']:
        print(f"  ⚠ Warnings       : {schema['warnings']}")


--- Type: WebSite (json-ld) ---
  Google supported : True
  Valid            : True
  Fields in raw    : ['@context', '@type', 'name', 'url', 'potentialAction', 'publisher']
  Present (spec)   : ['@type', 'url', 'name', 'potentialAction', 'potentialAction.@type', 'potentialAction.target', 'potentialAction.query-input']
  Missing required : []
  Missing recmd    : []

--- Type: LocalBusiness (json-ld) ---
  Google supported : True
  Valid            : True
  Fields in raw    : ['@context', '@type', 'name', 'description', 'url', 'image', 'address', 'telephone', 'openingHours', 'priceRange', 'sameAs']
  Present (spec)   : ['@type', 'name', 'address', 'address.@type', 'address.streetAddress', 'address.addressLocality', 'address.addressCountry', 'telephone', 'url', 'openingHours', 'priceRange', 'image']
  Missing required : []
  Missing recmd    : ['geo']
  ⚠ Warnings       : ["'@id' is missing — recommended for LocalBusiness to enable knowledge panel"]

--- Type: Organization (json-ld) --

In [ ]:
#| hide
#| eval: false
import requests
from bs4 import BeautifulSoup

r = requests.get("http://localhost:4321/sitemap-0.xml")
soup = BeautifulSoup(r.text, "xml")
urls = [loc.text for loc in soup.find_all("loc")]
print(f"Total URLs: {len(urls)}")
urls


Total URLs: 0


[]

Total URLs: 0


[]

In [ ]:
#| hide
#| eval: false
urls = ['http://localhost:4321',
        'http://localhost:4321/aazl-asth-alkwyt',
        'http://localhost:4321/about',
        'http://localhost:4321/anwaa-mwasyr-alghaz',
        'http://localhost:4321/astwanh-alghaz-faybr',
        'http://localhost:4321/blog',
        'http://localhost:4321/blog/2',
        'http://localhost:4321/blog/3',
        'http://localhost:4321/blog/4',
        'http://localhost:4321/blog/5',
        'http://localhost:4321/category/amdad_ghaz',
        'http://localhost:4321/category/amdad_ghaz/2',
        'http://localhost:4321/category/amdad_ghaz/3',
        'http://localhost:4321/category/amdad_ghaz/4',
        'http://localhost:4321/category/amdad_ghaz/5',
        'http://localhost:4321/category/awazl',
        'http://localhost:4321/contact',
        'http://localhost:4321/khzan-alghaz-almrkzy',
        'http://localhost:4321/landing/click-through',
        'http://localhost:4321/landing/lead-generation',
        'http://localhost:4321/landing/pre-launch',
        'http://localhost:4321/landing/product',
        'http://localhost:4321/landing/sales',
        'http://localhost:4321/pricing',
        'http://localhost:4321/privacy',
        'http://localhost:4321/services',
        'http://localhost:4321/shrkh-tmdyd-ghaz-balqsym',
        'http://localhost:4321/shrkh-tmdyd-ghaz-mrkzy-baldmam',
        'http://localhost:4321/syanh-afran-alghaz-balryadh',
        'http://localhost:4321/syanh-afran-alghaz-bjdh',
        'http://localhost:4321/syanh-afran-alghaz-bmkh',
        'http://localhost:4321/syanh-afran-ghaz-balmdynh-almnwrh',
        'http://localhost:4321/terms',
        'http://localhost:4321/tmdyd_alghaz_almrkzy_baltaef',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-alahsaa',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-balala',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-balmdnyh',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-balqtyf',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-balryadh',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-bmkh',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-bmsr',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-byshh',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-jdh',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-khmys-mshyt',
        'http://localhost:4321/tmdyd-alghaz-almrkzy-tbwk',
        'http://localhost:4321/tmdyd-ghaz-mrkzy-abw-zby',
        'http://localhost:4321/tmdyd-ghaz-mrkzy-babha',
        'http://localhost:4321/tmdyd-ghaz-mrkzy-balkhrj',
        'http://localhost:4321/tmdyd-ghaz-mrkzy-llmtaam',
        'http://localhost:4321/tmdyd-ghaz-mrkzy-mhayl-asyr',
        'http://localhost:4321/tmdyd-mwasyr-alghaz-llmtbkh']

In [ ]:
#| hide
#| eval: false
all_results = validate_pages(urls)


In [ ]:
#| hide
#| eval: false
for res in all_results:
    summary = res["summary"]
    status_icon = "✅" if summary["valid_count"] == summary["total_schemas"] and summary["total_schemas"] > 0 else (
        "⚠️" if summary["total_schemas"] > 0 else "❌")
    url_short = res["url"].replace("http://localhost:4321", "")
    print(
        f"{status_icon} {url_short or '/':<45} schemas:{summary['total_schemas']}  valid:{summary['valid_count']}  types:{summary['types_found']}")
    for schema in res["schemas_found"]:
        if schema["fields_missing_required"] or schema["warnings"]:
            print(
                f"     └─ {schema['type']}: missing_required={schema['fields_missing_required']}  warnings={schema['warnings']}")


✅ /                                             schemas:3  valid:3  types:['WebSite', 'LocalBusiness', 'Organization']
     └─ LocalBusiness: missing_required=[]  warnings=["'@id' is missing — recommended for LocalBusiness to enable knowledge panel"]
✅ /aazl-asth-alkwyt                             schemas:2  valid:2  types:['BlogPosting', 'FAQPage']
❌ /about                                        schemas:0  valid:0  types:[]
✅ /anwaa-mwasyr-alghaz                          schemas:1  valid:1  types:['BlogPosting']
✅ /astwanh-alghaz-faybr                         schemas:2  valid:2  types:['BlogPosting', 'FAQPage']
❌ /blog                                         schemas:0  valid:0  types:[]
❌ /blog/2                                       schemas:0  valid:0  types:[]
❌ /blog/3                                       schemas:0  valid:0  types:[]
❌ /blog/4                                       schemas:0  valid:0  types:[]
❌ /blog/5                                       schemas:0  valid:0  type